# 20 — Pull Worldwide Bureaucracy Indicators (WBI)

**Source:** World Bank Worldwide Bureaucracy Indicators  
**Provider:** World Bank  
**Access:** `wbgapi` Python package (database ID 63) — no API key required  
**Coverage:** ~180 economies, 2000–present (indicator-dependent)  
**Frequency:** Annual  

## What this notebook does
Pulls WBI indicators used as **coup-risk and state-capacity proxies** — primarily
public-sector wage and employment variables that proxy the government's ability to
retain military and civil service loyalty.

## Features pulled

| Feature | WBI Series | Why it matters |
|---|---|---|
| `wage_bill_gdp_pct` | `SG.GEN.TOTL.GD.ZS` | Govts that can't fund the wage bill face coup risk |
| `public_employment_pct` | `SL.PUB.EMPL.ZS` | Public sector share of total employment |
| `pub_priv_wage_ratio` | `SG.GEN.WPRS.SM.ZS` | Public-private wage premium (retention signal) |
| `health_workers_per_1k` | `SH.MED.NUMW.P3` | Service delivery capacity |
| `edu_public_exp_gdp` | `SE.XPD.TOTL.GD.ZS` | Redistributive capacity / legitimacy proxy |

## Required environment variables
```
ADLS_ACCOUNT_NAME  — Azure storage account name
ADLS_CONTAINER     — Container name (default: 'data')
```

In [ ]:
import os
import wbgapi
import pandas as pd
from datetime import datetime
from azure.identity import DefaultAzureCredential

## Configuration

In [ ]:
ADLS_ACCOUNT_NAME = os.environ["ADLS_ACCOUNT_NAME"]
ADLS_CONTAINER    = os.getenv("ADLS_CONTAINER", "data")
START_YEAR        = 1995
END_YEAR          = datetime.utcnow().year
RUN_DATE          = datetime.utcnow().strftime("%Y%m%d")

# WBI series codes (World Bank database 63)
WBI_SERIES = {
    "SG.GEN.TOTL.GD.ZS": "wage_bill_gdp_pct",
    "SL.PUB.EMPL.ZS":    "public_employment_pct",
    "SG.GEN.WPRS.SM.ZS": "pub_priv_wage_ratio",
    "SH.MED.NUMW.P3":    "health_workers_per_1k",
    "SE.XPD.TOTL.GD.ZS": "edu_public_exp_gdp",
}

print(f"Year range  : {START_YEAR}–{END_YEAR}")
print(f"Series      : {len(WBI_SERIES)} indicators")
print(f"ADLS target : abfss://{ADLS_CONTAINER}@{ADLS_ACCOUNT_NAME}.dfs.core.windows.net/raw/wbi/")
print(f"Run date    : {RUN_DATE}")

## ADLS helper

In [ ]:
credential = DefaultAzureCredential()
storage_options = {
    "account_name": ADLS_ACCOUNT_NAME,
    "credential": credential,
}

def adls_path(subpath: str) -> str:
    return (
        f"abfss://{ADLS_CONTAINER}@{ADLS_ACCOUNT_NAME}"
        f".dfs.core.windows.net/{subpath}"
    )

def write_parquet(df: pd.DataFrame, subpath: str) -> None:
    path = adls_path(subpath)
    df.to_parquet(path, storage_options=storage_options, index=False, engine="pyarrow")
    print(f"  Written {len(df):,} rows → {path}")

## Pull WBI indicators

`wbgapi` supports multiple databases; WBI lives in database 63.
We pull each series, pivot to wide format, and rename to descriptive column names.

In [ ]:
frames = []
for series_code, col_name in WBI_SERIES.items():
    try:
        df_s = wbgapi.data.DataFrame(
            series_code,
            time=range(START_YEAR, END_YEAR + 1),
            db=63,
            labels=True,
        ).reset_index()
        # wbgapi returns a multi-index (economy, time) — flatten
        if "economy" not in df_s.columns:
            df_s = df_s.rename_axis(index={"economy": "economy"}).reset_index()
        df_s = df_s.rename(columns={"economy": "iso3", "time": "year", series_code: col_name})
        df_s = df_s[["iso3", "year", col_name]]
        df_s["year"] = pd.to_numeric(df_s["year"], errors="coerce").astype("Int64")
        print(f"  {series_code:30s} → {col_name}: {len(df_s):,} obs, "
              f"{df_s[col_name].notna().sum():,} non-null")
        frames.append(df_s)
    except Exception as e:
        print(f"  WARNING: could not pull {series_code}: {e}")

# Merge on iso3 × year
from functools import reduce
df_wbi = reduce(lambda a, b: a.merge(b, on=["iso3", "year"], how="outer"), frames)
df_wbi = df_wbi.sort_values(["iso3", "year"]).reset_index(drop=True)

print(f"\nFinal panel shape: {df_wbi.shape}")
print(f"Countries  : {df_wbi['iso3'].nunique()}")
print(f"Year range : {df_wbi['year'].min()}–{df_wbi['year'].max()}")
df_wbi.head()

## Write to ADLS

In [ ]:
write_parquet(df_wbi, f"raw/wbi/{RUN_DATE}/wbi_panel.parquet")

## Summary

In [ ]:
print("=" * 55)
print("WBI pull complete")
print("=" * 55)
print(f"  Rows       : {len(df_wbi):,} country-years")
print(f"  Countries  : {df_wbi['iso3'].nunique()}")
print(f"  Year range : {df_wbi['year'].min()}–{df_wbi['year'].max()}")
print()
print("ADLS path written:")
print(f"  raw/wbi/{RUN_DATE}/wbi_panel.parquet")